# Chapter 16 &mdash; Counting Solutions: a BDD Does #SAT in One Pass

**Concept 16 of the Chapter 16 decomposition:** *Counting Solutions: a BDD Does #SAT in One Pass*

A solver returns one model. A BDD returns how many there are &mdash; a harder question, answered by one walk of a possibly huge diagram.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Counting-Solutions-Sharp-SAT/Concept-Counting-Solutions-Sharp-SAT.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


A CDCL solver answers *"is there a satisfying assignment, and here is one"*. A BDD
answers *"how many are there"*, which is strictly harder: **#SAT** is complete for
**#P**, a class sitting above NP. Knowing the count of certificates is more than
knowing whether one exists.

The count comes from a **single walk** of the diagram &mdash; each node combining the
counts of its two children &mdash; so it is linear **in the size of the diagram**.

That qualification is the entire story, and it is the same one as everywhere else in
this group: the hardness has not gone anywhere, it has moved into the size of the
structure.

Counting is also a cheap **diagnostic**. Two formulas over the same variables with
different counts are certainly not equivalent; equal counts prove nothing, but the
check costs one walk.

## 2. Definitions

### Count, and check the count

In [ ]:
f = bdd('Var_Order : a b c\nMain_Exp : (a | b) & (b | c)')
print(repr(f))

from itertools import product as _prod
brute = sum(1 for a, b, c in _prod([0, 1], repeat=3)
            if (a or b) and (b or c))
print()
print('bdd.count   :', f.count)
print('brute force :', brute)
assert f.count == brute

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;15.&nbsp;Why Converting CNF to DNF Is Not a Free Lunch](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-CNF-To-DNF-Is-No-Free-Lunch/Concept-CNF-To-DNF-Is-No-Free-Lunch.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;17.&nbsp;Graph Colouring as a Boolean Formula](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Graph-Colouring-As-A-Formula/Concept-Graph-Colouring-As-A-Formula.ipynb)&nbsp;&rarr;

---

## 3. Tests

**One model, or all of them.** This is the difference between what a solver hands back and what a diagram holds.

In [ ]:
print('a solver would return one of these:')
print('   ', f.models[0])
print()
print('the diagram holds all %d:' % f.count)
for m in f.models:
    print('   ', ''.join('%s=%d ' % (k, m[k]) for k in f.vars))

**Counting as a diagnostic.** Differing counts settle non-equivalence at once.

In [ ]:
g = bdd('Var_Order : a b c\nMain_Exp : (a | b) & (b | c) & (a | c)')
print('f: %d models    g: %d models' % (f.count, g.count))
print('different counts -> certainly not equivalent')
assert f.count != g.count
print()
h = bdd('Var_Order : a b c\nMain_Exp : ~(~(a | b) | ~(b | c))')
print('f: %d models    h: %d models' % (f.count, h.count))
print('equal counts -> proves nothing on its own; here they ARE equal as')
print('formulas, which needs the models compared, not just counted:')
print('   same model set?', f.models == h.models)

**Where the linear walk stops being cheap.** The count is linear in the diagram, and the diagram is what can explode &mdash; Concept 18.

In [ ]:
def wide(k):
    names = [['y%d_%d' % (i, j) for j in range(2)] for i in range(k)]
    flat  = [v for r in names for v in r]
    return ('Var_Order : ' + ' '.join(flat) + '\n'
            + '\n'.join('c%d = %s' % (i, ' | '.join(r))
                         for i, r in enumerate(names))
            + '\nMain_Exp : ' + ' & '.join('c%d' % i for i in range(k)))

print('%3s %6s %8s %14s' % ('k', 'vars', 'nodes', 'models'))
for k in range(1, 8):
    d = bdd(wide(k))
    print('%3d %6d %8d %14d' % (k, 2 * k, d.nodes, d.count))
print()
print('3^k models, counted without enumerating a single one of them.')
assert bdd(wide(7)).count == 3 ** 7

## 4. Exercises


1. `wide(k)` has $3^k$ models. Derive that by hand, then confirm the table.
2. Counting is linear in the diagram. Write down what the per-node rule must be for
   a node with variable $v$, and say where the $2^{\text{skipped}}$ factor for
   don't-care variables enters.
3. Two formulas over the same variables with the same count need not be equivalent.
   Construct a pair, using `f.models` to prove they differ.
4. A SAT solver can be made to count by adding a blocking clause per model found and
   re-solving. On `wide(7)`, how many solver calls would that take? Compare against
   one BDD walk.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16/Concept-Counting-Solutions-Sharp-SAT')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')